# 02 - Data Preprocessing

## 2.1 Overview

This notebook prepares the MVTec AD dataset for anomaly detection modeling.

Unlike a standard supervised image classification problem, the MVTec AD dataset is designed for industrial anomaly detection. The training set contains only normal samples, while the test set includes both normal and defective samples. For defective test images, pixel-level ground-truth masks are also provided.

The goal of this notebook is to create a reusable preprocessing pipeline that can support multiple anomaly detection models in later stages.

### Objectives

In this notebook, we will:

- Load training and test image paths
- Identify normal and defective samples
- Load ground-truth defect masks
- Resize images to a consistent input size
- Normalize image pixel values
- Prepare image transformations
- Apply training-time data augmentation
- Build reusable PyTorch dataset loaders
- Visualize processed images and masks
- Validate the final preprocessing pipeline

## 2.2 Import Libraries

This section imports the libraries required for image processing, dataset handling, visualization, and PyTorch model preparation.

PyTorch will be used for the modeling pipeline because it provides flexible tools for custom image datasets and is commonly used for modern anomaly detection approaches such as feature-based methods and PatchCore.

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

print("PyTorch version:", torch.__version__)

PyTorch version: 2.14.1


## 2.3 Configuration

To make the notebook easier to maintain and reuse, the main preprocessing parameters are defined in one place.

The image size is initially set to 224 × 224 pixels because this resolution is commonly used by pretrained convolutional neural networks such as ResNet and other ImageNet-based backbones.

The batch size can be adjusted later depending on available system memory and the requirements of the selected anomaly detection model.

In [13]:
# Dataset configuration
DATA_ROOT = Path("../1. Data/mvtec_anomaly_detection")

# Example category
CATEGORY = "bottle"

CATEGORY_PATH = DATA_ROOT / CATEGORY

# Image preprocessing configuration
IMAGE_SIZE = 224
BATCH_SIZE = 32

# Reproducibility
RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Dataset root:", DATA_ROOT.resolve())
print("Category:", CATEGORY)
print("Category path:", CATEGORY_PATH.resolve())
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)

Dataset root: /Users/junghanlee/Downloads/4. MVTec AD/1. Data/mvtec_anomaly_detection
Category: bottle
Category path: /Users/junghanlee/Downloads/4. MVTec AD/1. Data/mvtec_anomaly_detection/bottle
Image size: 224
Batch size: 32


## 2.4 Inspect Dataset Directory Structure

Before creating the preprocessing pipeline, we first inspect the directory structure of the selected MVTec AD category.

Each object category generally follows the structure below:

```text
category/
├── train/
│   └── good/
│
├── test/
│   ├── good/
│   ├── defect_type_1/
│   ├── defect_type_2/
│   └── ...
│
└── ground_truth/
    ├── defect_type_1/
    ├── defect_type_2/
    └── ...

In [14]:
def show_directory_structure(path, max_depth=3):
    """
    Display the directory structure up to a specified depth.
    """
    path = Path(path)

    for item in sorted(path.rglob("*")):
        relative_path = item.relative_to(path)

        if len(relative_path.parts) <= max_depth and item.is_dir():
            indent = "    " * (len(relative_path.parts) - 1)
            print(f"{indent}📁 {item.name}/")


show_directory_structure(CATEGORY_PATH)

📁 ground_truth/
    📁 broken_large/
    📁 broken_small/
    📁 contamination/
📁 test/
    📁 broken_large/
    📁 broken_small/
    📁 contamination/
    📁 good/
📁 train/
    📁 good/


## 2.5 Create Image Metadata

To simplify data handling throughout the project, we create a metadata table containing information about every training and test image.

For each image, the metadata stores:

- Image path
- Dataset split (`train` or `test`)
- Defect type
- Binary anomaly label

Normal (`good`) images receive a label of `0`, while defective images receive a label of `1`.

Creating a centralized metadata table makes it easier to inspect the dataset, build PyTorch datasets, perform evaluation, and analyze model performance by defect type.

In [17]:
def build_metadata(category_path):
    """
    Build a metadata DataFrame for one MVTec AD category.
    """
    records = []

    # -------------------------
    # Training images
    # -------------------------
    train_path = category_path / "train"

    for defect_dir in sorted(train_path.iterdir()):
        if not defect_dir.is_dir():
            continue

        defect_type = defect_dir.name

        for image_path in sorted(defect_dir.glob("*.png")):
            records.append({
                "path": image_path,
                "split": "train",
                "defect_type": defect_type,
                "label": 0
            })

    # -------------------------
    # Test images
    # -------------------------
    test_path = category_path / "test"

    for defect_dir in sorted(test_path.iterdir()):
        if not defect_dir.is_dir():
            continue

        defect_type = defect_dir.name
        label = 0 if defect_type == "good" else 1

        for image_path in sorted(defect_dir.glob("*.png")):
            records.append({
                "path": image_path,
                "split": "test",
                "defect_type": defect_type,
                "label": label
            })

    return pd.DataFrame(records)


metadata = build_metadata(CATEGORY_PATH)

metadata.head()

,path,split,defect_type,label
0,../1. Data/mvtec_anomaly_detection/bottle/trai...,train,good,0
1,../1. Data/mvtec_anomaly_detection/bottle/trai...,train,good,0
2,../1. Data/mvtec_anomaly_detection/bottle/trai...,train,good,0
3,../1. Data/mvtec_anomaly_detection/bottle/trai...,train,good,0
4,../1. Data/mvtec_anomaly_detection/bottle/trai...,train,good,0


In [18]:
print("Total images:", len(metadata))
print()

print("Images by split:")
print(metadata["split"].value_counts())

print("\nImages by defect type:")
print(metadata["defect_type"].value_counts())

print("\nImages by label:")
print(metadata["label"].value_counts())

Total images: 292

Images by split:
split
train    209
test      83
Name: count, dtype: int64

Images by defect type:
defect_type
good             229
broken_small      22
contamination     21
broken_large      20
Name: count, dtype: int64

Images by label:
label
0    229
1     63
Name: count, dtype: int64


### Metadata Interpretation

The metadata confirms the fundamental structure of the anomaly detection problem.

The training set contains only normal (`good`) samples, while the test set contains both normal and anomalous samples.

This means that the model cannot simply learn a supervised mapping between known defect classes and labels. Instead, it must learn a representation of normal appearance and identify samples that deviate from that representation.

## 2.6 Map Ground-Truth Masks

MVTec AD provides pixel-level ground-truth masks for anomalous test images.

Each mask identifies the exact region of the image where a defect occurs:

- White pixels represent defective regions.
- Black pixels represent normal regions.

Normal (`good`) test images do not have ground-truth masks because no defective region exists.

These masks will not be used to train the initial anomaly detection model. Instead, they will become important during evaluation when measuring how accurately the model localizes defects.

In [19]:
def get_mask_path(image_path, category_path, defect_type):
    """
    Return the corresponding ground-truth mask path
    for an anomalous test image.
    """
    if defect_type == "good":
        return None

    mask_name = f"{image_path.stem}_mask.png"

    mask_path = (
        category_path
        / "ground_truth"
        / defect_type
        / mask_name
    )

    return mask_path if mask_path.exists() else None

In [20]:
metadata["mask_path"] = metadata.apply(
    lambda row: (
        get_mask_path(
            row["path"],
            CATEGORY_PATH,
            row["defect_type"]
        )
        if row["split"] == "test" and row["label"] == 1
        else None
    ),
    axis=1
)

metadata.head()

,path,split,defect_type,label,mask_path
0,../1. Data/mvtec_anomaly_detection/bottle/trai...,train,good,0,None
1,../1. Data/mvtec_anomaly_detection/bottle/trai...,train,good,0,None
2,../1. Data/mvtec_anomaly_detection/bottle/trai...,train,good,0,None
3,../1. Data/mvtec_anomaly_detection/bottle/trai...,train,good,0,None
4,../1. Data/mvtec_anomaly_detection/bottle/trai...,train,good,0,None


In [21]:
anomalous_test = metadata[
    (metadata["split"] == "test") &
    (metadata["label"] == 1)
]

print("Anomalous test images:", len(anomalous_test))
print(
    "Ground-truth masks found:",
    anomalous_test["mask_path"].notna().sum()
)

Anomalous test images: 63
Ground-truth masks found: 63
